In [2]:
import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, classification_report

In [10]:
#import files

gt = pd.read_csv("/Users/saharshbhave/Downloads/Ground_Truth.csv")
pred = pd.read_csv("/Users/saharshbhave/Downloads/RB_GPT_Pred.csv")

#Column names
causality_col = "Causality"
relation_col = "Relation_Type"

In [11]:
#Relation Labels

Relation_Labels = [
    "Mechanistic/Theory",
    "Measurement/Method",
    "Description",
    "Correlation",
    "Hypothesis",
    "Association"
]

In [12]:
#Validate

if len(gt) != len(pred):
    raise ValueError(f"Row mismatch: gt{len(gt)}, pred{len(pred)}")

#Normalize values
def normalize_causality(x):
    if pd.isna(x):
        return ""
    x = str(x).strip().lower()
    if x in ["Yes", "yes"]:
        return "yes"
    if x in ["No", "no"]:
        return "no"
    
def normalize_relation(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()


gt[causality_col] = gt[causality_col].apply(normalize_causality)
pred[causality_col] = pred[causality_col].apply(normalize_causality)

gt[relation_col] = gt[relation_col].apply(normalize_relation)
pred[relation_col] = pred[relation_col].apply(normalize_relation)

In [13]:
#Causality Metrics

y_true_c = gt[causality_col]
y_pred_c = pred[causality_col]

causality_labels = ["yes", "no"]

cm_c = confusion_matrix(y_true_c, y_pred_c, labels=causality_labels)
print(confusion_matrix)
print(pd.DataFrame(cm_c, index=causality_labels, columns=causality_labels))

f1_c = f1_score(y_true_c, y_pred_c, pos_label="yes", average="binary")
weighted_f1_c = f1_score(y_true_c, y_pred_c, average="weighted")

print(f"\nF1 Score: {f1_c:.4f}")
print(f"Weighted F1 Score: {weighted_f1_c:.4f}")

print("\nClassification Report:")
print(classification_report(y_true_c, y_pred_c, labels=causality_labels, zero_division=0))

<function confusion_matrix at 0x1181256c0>
     yes  no
yes   32   2
no    13  13

F1 Score: 0.8101
Weighted F1 Score: 0.7339

Classification Report:
              precision    recall  f1-score   support

         yes       0.71      0.94      0.81        34
          no       0.87      0.50      0.63        26

    accuracy                           0.75        60
   macro avg       0.79      0.72      0.72        60
weighted avg       0.78      0.75      0.73        60



In [14]:
# Relation Type Metrics
#only where causality = no

mask_no = gt[causality_col].astype(str).str.strip().str.lower() == "no"

y_true_r = gt.loc[mask_no, relation_col].copy()
y_pred_r = pred.loc[mask_no, relation_col].copy()

# normalize
y_true_r = y_true_r.fillna("").astype(str).str.strip().str.lower()
y_pred_r = y_pred_r.fillna("").astype(str).str.strip().str.lower()

print("Number of GT 'no' rows:", len(y_true_r))
print("Unique y_true relation values:", y_true_r.unique())
print("Unique y_pred relation values:", y_pred_r.unique())

# replace blank strings with placeholder
y_true_r = y_true_r.replace("", "missing_prediction")
y_pred_r = y_pred_r.replace("", "missing_prediction")

# build labels from actual data instead of fixed list
relation_labels = sorted(set(y_true_r) | set(y_pred_r))

print("Labels used for scoring:", relation_labels)

if len(y_true_r) == 0:
    print("No rows found where ground truth causality = 'no'. Relation metrics cannot be computed.")
else:
    cm_r = confusion_matrix(y_true_r, y_pred_r, labels=relation_labels)
    print("Confusion Matrix:")
    print(pd.DataFrame(cm_r, index=relation_labels, columns=relation_labels))

    f1_macro_r = f1_score(y_true_r, y_pred_r, average="macro")
    f1_weighted_r = f1_score(y_true_r, y_pred_r, average="weighted")

    print(f"\nF1 Score (Macro): {f1_macro_r:.4f}")
    print(f"Weighted F1 Score: {f1_weighted_r:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_true_r, y_pred_r, labels=relation_labels, zero_division=0))

Number of GT 'no' rows: 26
Unique y_true relation values: ['mechanistic / theory' 'measurement/method' 'description' 'hypothesis']
Unique y_pred relation values: ['description' '' 'measurement/method' 'association']
Labels used for scoring: ['association', 'description', 'hypothesis', 'measurement/method', 'mechanistic / theory', 'missing_prediction']
Confusion Matrix:
                      association  description  hypothesis  \
association                     0            0           0   
description                     0            0           0   
hypothesis                      0            0           0   
measurement/method              1            0           0   
mechanistic / theory            1            3           0   
missing_prediction              0            0           0   

                      measurement/method  mechanistic / theory  \
association                            0                     0   
description                            1                     